In [2]:
from massspecgym.data.data_module import MassSpecDataModule
from massspecgym.data.datasets import RetrievalDataset, MassSpecDataset
from massspecgym.data.transforms import InMemCachedMolTransform
from chemembed_transforms import (
    ChemEmbedSpecTransform, ChemEmbedMolTransform,
    ChemBERTaMolTransform, MoLFormerMolTransform,
    chemberta_factory, molformer_factory, mol2vec_factory
)
from pathlib import Path
from ann_model_massspecgym import AnnRetrieval
from pytorch_lightning import Trainer
from massspecgym.models.base import Stage
import pandas as pd
import pickle
import tqdm as tqdm_module

c:\Users\aa\anaconda3\envs\massspecgym\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Failed to find the pandas get_adjustment() function to patch
Failed to patch pandas - PandasTools will have limited functionality


In [3]:
######### Embedding selection ##########

# Mol2Vec (300 dims)
# mol_transform_factory = mol2vec_factory
# embedding_name = "mol2vec"
# mol_embedding_dim = 300

# ChemBERTa (768 dims)
mol_transform_factory = chemberta_factory
embedding_name = "chemberta"
mol_embedding_dim = 768

# MoLFormer (768 dims)
# mol_transform_factory = molformer_factory
# embedding_name = "molformer"
# mol_embedding_dim = 768

####################

selected_molecular_embedding = InMemCachedMolTransform(
    cache_pth=f"data/{embedding_name}_cache.pkl",
    mol_transform_factory=mol_transform_factory,
    verbose=True,
)

# Train dataset, without reference candidates
train_dataset = MassSpecDataset(
    spec_transform=ChemEmbedSpecTransform(),
    mol_transform=selected_molecular_embedding,
)
data_module_train = MassSpecDataModule(dataset=train_dataset, batch_size=32)
data_module_train.prepare_data()
data_module_train.setup()

# Construir cache en batches con GPU (solo la primera vez)
cache_path = Path(f"data/{embedding_name}_cache.pkl")
if not cache_path.exists():
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    transform = mol_transform_factory()
    smiles_sorted = sorted(set(train_dataset.metadata["smiles"].tolist()))
    cache = {}
    for i in tqdm_module.tqdm(range(0, len(smiles_sorted), 128), desc=f"Building {embedding_name} cache"):
        batch = smiles_sorted[i:i+128]
        embeddings = transform.from_smiles(batch)
        for smi, emb in zip(batch, embeddings):
            cache[smi] = emb
    with open(cache_path, "wb") as f:
        pickle.dump(cache, f, protocol=pickle.HIGHEST_PROTOCOL)

selected_molecular_embedding._load_cache()

####################

# Init model
model = AnnRetrieval(
    mol_embedding_dim=mol_embedding_dim,
    log_only_loss_at_stages=[Stage.TRAIN, Stage.VAL]
)

# Init trainer
trainer = Trainer(
    accelerator="auto",
    max_epochs=15,
    log_every_n_steps=1
)

# trainer.validate(model, datamodule=data_module_train)
# trainer.fit(model, datamodule=data_module_train)
# trainer.save_checkpoint(f"ANN_trained_{embedding_name}.ckpt")

c:\Users\aa\anaconda3\envs\massspecgym\Lib\site-packages\huggingface_hub\file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Initializing InMemCachedMolTransform for ChemBERTaMolTransform. Loading cache from data\chemberta_cache.pkl...


GPU available: False, used: False
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
c:\Users\aa\anaconda3\envs\massspecgym\Lib\site-packages\pytorch_lightning\trainer\connectors\logger_connector\logger_connector.py:75: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `pytorch_lightning` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default


In [4]:
######### Cargar modelo entrenado + extender cache para test ##########
# ChemBERTa:
mol_transform_factory = chemberta_factory
embedding_name = "chemberta"
mol_embedding_dim = 768

# MoLFormer:
# mol_transform_factory = molformer_factory
# embedding_name = "molformer"
# mol_embedding_dim = 768

cache_path = Path(f"data/{embedding_name}_cache.pkl")

# Cargar candidatos para obtener todos los SMILES de test
test_dataset_tmp = RetrievalDataset(candidates_pth='standard')
bonus_dataset_tmp = RetrievalDataset(candidates_pth='bonus')
test_candidate_smiles = {smi for cands in test_dataset_tmp.candidates.values() for smi in cands}
bonus_candidate_smiles = {smi for cands in bonus_dataset_tmp.candidates.values() for smi in cands}
all_candidate_smiles = list(test_candidate_smiles | bonus_candidate_smiles)

# Cargar cache existente y calcular solo los SMILES que falten
with open(cache_path, "rb") as f:
    existing_cache = pickle.load(f)

missing_smiles = sorted({s for s in all_candidate_smiles if s not in existing_cache})
print(f"SMILES ya cacheados: {len(existing_cache)}, faltan: {len(missing_smiles)}")

if missing_smiles:
    transform = mol_transform_factory()
    for i in tqdm_module.tqdm(range(0, len(missing_smiles), 128), desc=f"Extending {embedding_name} cache"):
        batch = missing_smiles[i:i+128]
        embeddings = transform.from_smiles(batch)
        for smi, emb in zip(batch, embeddings):
            existing_cache[smi] = emb
    with open(cache_path, "wb") as f:
        pickle.dump(existing_cache, f, protocol=pickle.HIGHEST_PROTOCOL)
    print("Cache extendido y guardado.")

# Cargar el transform con el cache completo
selected_molecular_embedding = InMemCachedMolTransform(
    cache_pth=cache_path,
    mol_transform_factory=mol_transform_factory,
    verbose=True,
)

# Cargar modelo entrenado
model = AnnRetrieval.load_from_checkpoint(
    f"ANN_trained_{embedding_name}.ckpt",
    mol_embedding_dim=mol_embedding_dim,
)
trainer = Trainer(accelerator="auto")


SMILES ya cacheados: 31602, faltan: 11742886


c:\Users\aa\anaconda3\envs\massspecgym\Lib\site-packages\huggingface_hub\file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Extending chemberta cache:   0%|          | 19/91742 [07:56<638:45:43, 25.07s/it]


KeyboardInterrupt: 

In [ ]:
# Standard test 
test_dataset = RetrievalDataset(
    spec_transform=ChemEmbedSpecTransform(),
    mol_transform=selected_molecular_embedding,
)
data_module_test = MassSpecDataModule(dataset=test_dataset, batch_size=32)
data_module_test.setup(stage="test")

test_results = trainer.test(model, datamodule=data_module_test)
results_df = pd.DataFrame(test_results)
results_df.to_csv(f"masspecgym_ann_results_{embedding_name}.csv", index=False)

In [ ]:
# BONUS test with candidates (same molecular formula)
test_dataset_bonus = RetrievalDataset(
    spec_transform=ChemEmbedSpecTransform(),
    mol_transform=selected_molecular_embedding,
    candidates_pth='bonus'
)
data_module_test_bonus = MassSpecDataModule(dataset=test_dataset_bonus, batch_size=32)
data_module_test_bonus.setup(stage="test")

test_results_bonus = trainer.test(model, datamodule=data_module_test_bonus)
results_df_bonus = pd.DataFrame(test_results_bonus)
results_df_bonus.to_csv(f"masspecgym_ann_results_bonus_{embedding_name}.csv", index=False)